# Stage C — Build Basin-Level **Daily** Targets & Join ERA5 **Daily** Features

**Purpose**
Create a clean, daily target series of **basin discharge (m³/s)** for the 3 modeled basins and join it with **ERA5 basin-level daily features**. This notebook **uses an existing daily ERA5 parquet** from Stage B.

---

## What this notebook does

1. Resolve project root (Git/heuristic).
2. Load **station → basin name** selections and merge with **basin name → numeric `basin_id`** lookup.
3. Read **per-station daily discharge** parquets, attach `basin_id`, and collapse to **one target per basin per day**.
4. Load **ERA5 basin daily** features, subset to the **3 basins** and the **target date window**, optionally prune variables.
5. Join features ↔ targets and write compact artifacts.

---

## Inputs

* **Per-station daily discharge** (from EDA)
  `data/basin_discharge/processed/station=<slug>.parquet`
  *Columns used:* `station_name, datetime_local, discharge_cms, qc_any`
* **Station → Basin (source of truth)**
  `data/modeling/targets/meta/station_to_basin_name.csv`
  *(station\_name, use\_for\_model, basin\_name)*
* **Basin name → ID lookup**
  `data/boundaries/processed/basin_lookup.csv`
  *(basin\_name, basin\_id\[, basin\_slug])*
* **ERA5 (already daily, basin level)**
  `data/era5/era5_aligned/basin_level/era5_basin_daily.parquet`
  *(must include `basin_id` and `date_local` or `datetime_local`)*

> Optional: set an `ERA5_KEEP` whitelist in the notebook to keep only selected ERA5 variables.

---

## Outputs

* **Targets (daily, basin level)**
  `data/modeling/targets/targets_discharge_basin_daily.parquet`
  *Columns:* `basin_id, basin_name, date_local, discharge_cms, qc_any`
* **Joined features + target (daily)**
  `data/modeling/targets/train_basin_daily.parquet`
  *Columns:* `basin_id, date_local, discharge_cms, qc_any, <ERA5 features…>`

---

## Assumptions & conventions

* **Time zone:** all timestamps are Bhutan local; we normalize to **naive midnight** `date_local`.
* **Units:** `discharge_cms` is **m³/s**; ERA5 remains in native units.
* **QC:** we **carry** `qc_any`; flagged values aren’t dropped (filter later if desired).
* **One station per modeled basin:** exactly **3** stations have `use_for_model=True`, each mapping to **one** basin (the notebook errors if 0 or >1 per basin).
* **ERA5 coverage:** ERA5 may include extra basins/dates/variables; we subset to the **3 basins** and the **target date span** (and optionally prune variables).

---

## High-level flow

1. **Map stations → basins:**
   `station_to_basin_name.csv` ⟶ merge with `basin_lookup.csv` ⟶ add `basin_id`.
2. **Targets:**
   Load station parquets ⟶ derive `date_local` ⟶ attach `basin_id` ⟶ group by (`basin_id`, `date_local`) to get `discharge_cms` (mean if duplicates) and `qc_any` (max).
3. **ERA5 features:**
   Load `era5_basin_daily.parquet` ⟶ keep modeled basins ⟶ trim to target date range ⟶ optional variable pruning.
4. **Join:**
   Left-join on (`basin_id`, `date_local`) ⟶ write **targets** and **train** artifacts.

---

## Quick QA printed by the notebook

* Selected stations with their `basin_id`s.
* Head of targets and ERA5 daily frames.
* Join stats: total rows and rows with all features missing (should be ≈0).

## Resolve Project Root

In [1]:
# === Resolve Project Root ===
from pathlib import Path
import subprocess

def get_project_root(max_up=6):
    try:
        root = subprocess.check_output(["git","rev-parse","--show-toplevel"], text=True).strip()
        if root:
            return Path(root)
    except Exception:
        pass
    p = Path.cwd()
    for _ in range(max_up):
        if (p/"data").exists() and (p/"code").exists():
            return p
        if (p/".git").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = get_project_root()
print("Project root:", PROJECT_ROOT)


Project root: /Users/liuq13/bhutan_climate_modeling


## Paths & Config

In [2]:
# === Paths & Config ===
import pandas as pd
import numpy as np
from pathlib import Path
import glob

# Inputs
STATION_DAILY_DIR      = PROJECT_ROOT / "data/basin_discharge/processed"           # station=<slug>.parquet
STATION_TO_BASIN_CSV   = PROJECT_ROOT / "data/modeling/targets/meta/station_to_basin_name.csv"
BASIN_LOOKUP_CSV       = PROJECT_ROOT / "data/boundaries/processed/basin_lookup.csv"
ERA5_DAILY_PARQUET     = PROJECT_ROOT / "data/era5/era5_aligned/basin_level/era5_basin_daily.parquet"  # era5 daily data at basin level

# Outputs
TARGETS_DAILY_PARQUET  = PROJECT_ROOT / "data/modeling/targets/targets_discharge_basin_daily.parquet"
JOINED_DAILY_PARQUET   = PROJECT_ROOT / "data/modeling/targets/train_basin_daily.parquet"

# Optional: if you want to prune ERA5 variables now, list them here (keeps IDs & date automatically)
ERA5_KEEP = None
# Example:
# ERA5_KEEP = ["total_precipitation","runoff","surface_runoff","sub_surface_runoff",
#              "potential_evaporation","temperature","dewpoint",
#              "soil_temperature_level_1","snow_depth","snowmelt"]


## Station → Basin mapping

In [3]:
# === Station → Basin mapping ===
st2name = pd.read_csv(STATION_TO_BASIN_CSV)           # station_name, use_for_model, basin_name
lookup  = pd.read_csv(BASIN_LOOKUP_CSV)               # basin_id, basin_name, (basin_slug optional)

mapping = (st2name.merge(lookup, on="basin_name", how="left")
                 .query("use_for_model == True")
                 .copy())

if mapping["basin_id"].isna().any():
    missing = mapping[mapping["basin_id"].isna()][["station_name","basin_name"]]
    raise ValueError(f"Basin name not found in lookup for:\n{missing}")

print("Chosen stations → basins:")
display(mapping[["station_name","basin_name","basin_id"]])

Chosen stations → basins:


,station_name,basin_name,basin_id
0,Kurjey (Chamkharchhu),Mangdechhu,3.0
1,Lungtenphu (Wangchhu),Wangchhu,6.0
3,Wangdi rapid (Punatshangchhu),Punatsangchhu,8.0


## Build daily targets per basin from your per-station daily parquets

In [4]:
# === Targets: load per-station daily, attach basin_id, collapse to one series per basin ===

def slugify_simple(s: str) -> str:
    return s.lower().strip().replace(" ", "_")

by_station = []
for st in mapping["station_name"]:
    # try direct slug match first
    slug = slugify_simple(st)
    cands = list(STATION_DAILY_DIR.glob(f"station={slug}*.parquet"))
    if not cands:
        # fallback: scan all and check content
        for p in STATION_DAILY_DIR.glob("station=*.parquet"):
            dfp = pd.read_parquet(p, columns=["station_name"])
            if dfp["station_name"].iloc[0] == st:
                cands = [p]; break
    if not cands:
        raise FileNotFoundError(f"Daily parquet not found for station: {st}")
    p = cands[0]
    df = pd.read_parquet(p)
    # expected columns: station_name, datetime_local, discharge_cms, qc_any (others ok)
    need = {"station_name","datetime_local","discharge_cms"}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"{p} missing columns: {missing}")
    df["date_local"] = pd.to_datetime(df["datetime_local"]).dt.floor("D")
    by_station.append(df[["station_name","date_local","discharge_cms","qc_any"]])

st_daily = pd.concat(by_station, ignore_index=True)

# attach basin_id/name and sanity-check one station per basin
st_daily = st_daily.merge(mapping[["station_name","basin_id","basin_name"]],
                          on="station_name", how="left")
chk = st_daily[["station_name","basin_id"]].drop_duplicates()
if (chk["basin_id"].value_counts() > 1).any():
    raise ValueError("A basin maps to >1 station after filtering; fix your template.")

# reduce to 1 row per basin/day (mean handles any accidental duplicates)
targets_daily = (st_daily.groupby(["basin_id","basin_name","date_local"], as_index=False)
                        .agg(discharge_cms=("discharge_cms","mean"),
                             qc_any=("qc_any","max"))
                 .sort_values(["basin_id","date_local"])
                 .reset_index(drop=True))

print("Targets daily preview:")
display(targets_daily.head(10))


Targets daily preview:


,basin_id,basin_name,date_local,discharge_cms,qc_any
0,3.0,Mangdechhu,2000-01-01,14.783,False
1,3.0,Mangdechhu,2000-01-02,14.620,False
2,3.0,Mangdechhu,2000-01-03,14.565,False
3,3.0,Mangdechhu,2000-01-04,14.031,False
4,3.0,Mangdechhu,2000-01-05,14.032,False
5,3.0,Mangdechhu,2000-01-06,14.189,False
6,3.0,Mangdechhu,2000-01-07,14.296,False
7,3.0,Mangdechhu,2000-01-08,14.565,False
8,3.0,Mangdechhu,2000-01-09,14.350,False
9,3.0,Mangdechhu,2000-01-10,14.565,False


In [5]:
# === Load ERA5 daily (already prepared) and subset ===
e = pd.read_parquet(ERA5_DAILY_PARQUET)

# Ensure a proper Timestamp day column
if "date_local" in e.columns:
    e["date_local"] = pd.to_datetime(e["date_local"], errors="coerce").dt.tz_localize(None).dt.normalize()
elif "datetime_local" in e.columns:
    e["date_local"] = pd.to_datetime(e["datetime_local"], errors="coerce").dt.tz_localize(None).dt.normalize()
    e = e.drop(columns=["datetime_local"])
else:
    raise ValueError("ERA5 daily parquet must have 'date_local' or 'datetime_local'.")

# Keep only modeled basins
e = e[e["basin_id"].isin(mapping["basin_id"].unique())].copy()

# Optional pruning of variables
if ERA5_KEEP is not None:
    keep = ["basin_id","date_local"] + [c for c in ERA5_KEEP if c in e.columns]
    e = e[keep]

# --- Fix: make targets' dates Timestamps too ---
targets_daily["date_local"] = pd.to_datetime(targets_daily["date_local"], errors="coerce").dt.tz_localize(None).dt.normalize()

# Trim to target date window (both sides now Timestamp, comparable)
start = targets_daily["date_local"].min()
end   = targets_daily["date_local"].max()
e = e[(e["date_local"] >= start) & (e["date_local"] <= end)].copy()

print("ERA5 daily preview:")
display(e.head())


ERA5 daily preview:


,basin_id,date_local,temperature,dewpoint,wind_u,wind_v,potential_evaporation,runoff,snow_depth,snowmelt,soil_temperature,sub_surface_runoff,surface_runoff,solar_radiation,precipitation,n_cells,low_coverage
21501,3.0,1991-04-22,4.977037,275.264827,0.223310,0.405829,-0.000256,0.000670,0.091980,0.000282,278.790527,0.000162,0.000508,2.669388e+06,0.001312,11,False
21502,3.0,1991-04-23,5.864305,276.764485,0.240125,0.136637,-0.000298,0.001176,0.091298,0.000363,279.128706,0.000163,0.001013,2.982208e+06,0.002204,11,False
21503,3.0,1991-04-24,6.002400,276.720788,0.400158,-0.004225,-0.000290,0.001409,0.090082,0.000322,279.054998,0.000165,0.001243,3.243782e+06,0.002312,11,False
21504,3.0,1991-04-25,5.003354,276.180910,0.199404,0.122593,-0.000319,0.000498,0.089339,0.000321,279.124212,0.000170,0.000327,3.223820e+06,0.000747,11,False
21505,3.0,1991-04-26,5.906758,276.869664,0.195405,0.217584,-0.000338,0.000730,0.088612,0.000363,279.410822,0.000176,0.000554,3.711197e+06,0.001180,11,False


In [6]:
# === Join features ↔ target (daily) and write outputs ===
# Identify feature columns (numeric, excluding IDs & date)
exclude = {"basin_id","date_local"}
num_cols = [c for c in e.columns if c not in exclude and pd.api.types.is_numeric_dtype(e[c])]

train_daily = (targets_daily
               .merge(e, on=["basin_id","date_local"], how="left")
               .sort_values(["basin_id","date_local"])
               .reset_index(drop=True))

# Quick QA
n_all   = len(train_daily)
n_missF = train_daily[num_cols].isna().all(axis=1).sum() if num_cols else 0
print(f"Joined rows: {n_all}  |  rows with all ERA5 features missing: {n_missF}")

# Write
TARGETS_DAILY_PARQUET.parent.mkdir(parents=True, exist_ok=True)
JOINED_DAILY_PARQUET.parent.mkdir(parents=True, exist_ok=True)

targets_daily.to_parquet(TARGETS_DAILY_PARQUET, index=False)
train_daily.to_parquet(JOINED_DAILY_PARQUET, index=False)

print("Wrote targets:", TARGETS_DAILY_PARQUET)
print("Wrote joined:",  JOINED_DAILY_PARQUET)
display(train_daily.head())


Joined rows: 29840  |  rows with all ERA5 features missing: 0
Wrote targets: /Users/liuq13/bhutan_climate_modeling/data/modeling/targets/targets_discharge_basin_daily.parquet
Wrote joined: /Users/liuq13/bhutan_climate_modeling/data/modeling/targets/train_basin_daily.parquet


,basin_id,basin_name,date_local,discharge_cms,qc_any,temperature,dewpoint,wind_u,wind_v,potential_evaporation,runoff,snow_depth,snowmelt,soil_temperature,sub_surface_runoff,surface_runoff,solar_radiation,precipitation,n_cells,low_coverage
0,3.0,Mangdechhu,2000-01-01,14.783,False,-2.497898,265.813408,0.095126,-0.488079,-0.000222,0.000143,0.034493,0.000080,274.263428,0.000114,0.000029,2.368407e+06,0.000018,11,False
1,3.0,Mangdechhu,2000-01-02,14.620,False,-2.066571,265.773514,0.254713,-0.419430,-0.000248,0.000145,0.033568,0.000075,274.279602,0.000114,0.000031,2.494330e+06,0.000029,11,False
2,3.0,Mangdechhu,2000-01-03,14.565,False,-3.107548,266.192237,0.277529,-0.309926,-0.000225,0.000140,0.033217,0.000052,274.151822,0.000113,0.000028,2.318063e+06,0.000090,11,False
3,3.0,Mangdechhu,2000-01-04,14.031,False,-3.349576,265.113591,0.199192,-0.453193,-0.000241,0.000128,0.033088,0.000042,274.176020,0.000112,0.000017,2.486289e+06,0.000015,11,False
4,3.0,Mangdechhu,2000-01-05,14.032,False,-2.330868,266.624595,0.226005,-0.230016,-0.000214,0.000122,0.032933,0.000027,274.346547,0.000110,0.000011,2.144902e+06,0.000050,11,False
